In [ ]:
# ==============================================================================
# Mount Google Drive & Import Libraries
# ==============================================================================
import os
import joblib
import numpy as np
import pandas as pd
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# ==============================================================================
# Define File Paths & Directories
# ==============================================================================
BASE_DIR = '/content/drive/MyDrive/WESAD_Model'
INPUT_DIR = os.path.join(BASE_DIR, 'InputData')
OUTPUT_FILES_DIR = os.path.join(BASE_DIR, 'OutPutFiles')
OUTPUT_MODEL_DIR = os.path.join(BASE_DIR, 'OutPutModel')

# Create output directories if they don't exist
os.makedirs(OUTPUT_FILES_DIR, exist_ok=True)
os.makedirs(OUTPUT_MODEL_DIR, exist_ok=True)

# File Paths
INPUT_CSV_PATH = os.path.join(INPUT_DIR, 'WESAD_FINAL_NSRI_30sec_50overlap_CLEAN.csv')
MODEL_SAVE_PATH = os.path.join(OUTPUT_MODEL_DIR, 'ExtraTreeWESADModel01.pkl')
TEST_RESULTS_PATH = os.path.join(OUTPUT_FILES_DIR, 'ExtraTreeWesadModelTestResults.csv')
PRED_RESULTS_PATH = os.path.join(OUTPUT_FILES_DIR, 'ExtraTreeWesadModelPredResults01.csv')

print(f"Input Dataset:  {INPUT_CSV_PATH}")
print(f"Model Save Path: {MODEL_SAVE_PATH}")
print(f"Test Results:    {TEST_RESULTS_PATH}")
print(f"Pred Results:    {PRED_RESULTS_PATH}")

Mounted at /content/drive
Input Dataset:  /content/drive/MyDrive/WESAD_Model/InputData/WESAD_FINAL_NSRI_30sec_50overlap_CLEAN.csv
Model Save Path: /content/drive/MyDrive/WESAD_Model/OutPutModel/ExtraTreeWESADModel01.pkl
Test Results:    /content/drive/MyDrive/WESAD_Model/OutPutFiles/ExtraTreeWesadModelTestResults.csv
Pred Results:    /content/drive/MyDrive/WESAD_Model/OutPutFiles/ExtraTreeWesadModelPredResults01.csv


In [ ]:
# Standalone Predict Function
def run_standalone_inference(input_csv, model_path, output_results_csv):
  """Loads the saved Extra Trees .pkl model, runs inference across the entire dataset,

  and exports the results.
  """
  # 1. Load model
  model = joblib.load(model_path)

  # 2. Load dataset
  data = pd.read_csv(input_csv)

  # 3. Align features
  features = [
      c
      for c in model.feature_names_in_
      if c in data.columns
  ]
  X_in = data[features].copy()
  X_in = X_in.replace([np.inf, -np.inf], np.nan)
  X_in = X_in.fillna(X_in.mean())

  # 4. Predict
  preds = model.predict(X_in)
  probas = model.predict_proba(X_in)

  # 5. Compile Results
  out_df = data.copy()
  out_df["Predicted_Target"] = preds
  out_df["Stress_Probability"] = probas[:, 1]
  out_df["Prediction_Confidence"] = np.max(probas, axis=1) * 100.0

  if "stress_target" in out_df.columns:
    out_df["Actual_Label"] = out_df["stress_target"].map(
        {0: "Not Target", 1: "Target"}
    )
    out_df["Prediction_Status"] = np.where(
        out_df["stress_target"] == out_df["Predicted_Target"],
        "Correct",
        "Incorrect",
    )

  out_df["Predicted_Label"] = out_df["Predicted_Target"].map(
      {0: "Not Target", 1: "Target"}
  )
  out_df["Model_Used"] = "Extra Trees"

  out_df.to_csv(output_results_csv, index=False)
  print(
      f"[SUCCESS] Standalone predictions exported to: {output_results_csv} "
      f"(Total Rows: {len(out_df)})"
  )
  return out_df


# Run standalone inference on the full dataset
full_pred_df = run_standalone_inference(
    input_csv=INPUT_CSV_PATH,
    model_path=MODEL_SAVE_PATH,
    output_results_csv=PRED_RESULTS_PATH,
)

display(
    full_pred_df[[
        "subject",
        "window_id",
        "stress_target",
        "Predicted_Target",
        "Stress_Probability",
        "Prediction_Confidence",
        "Prediction_Status",
    ]].head(10)
)

[SUCCESS] Standalone predictions exported to: /content/drive/MyDrive/WESAD_Model/OutPutFiles/ExtraTreeWesadModelPredResults01.csv (Total Rows: 2878)


,subject,window_id,stress_target,Predicted_Target,Stress_Probability,Prediction_Confidence,Prediction_Status
0,S10,6,0,0,0.000000,100.000000,Correct
1,S10,7,0,0,0.000000,100.000000,Correct
2,S10,8,0,0,0.000000,100.000000,Correct
3,S10,9,0,0,0.000000,100.000000,Correct
4,S10,10,0,0,0.086667,91.333333,Correct
5,S10,11,0,0,0.000000,100.000000,Correct
6,S10,12,0,0,0.000000,100.000000,Correct
7,S10,13,0,0,0.000000,100.000000,Correct
8,S10,14,0,0,0.040000,96.000000,Correct
9,S10,15,0,0,0.000000,100.000000,Correct
